<a href="https://colab.research.google.com/github/nainikadevireddy/JohnsHopkinsAI/blob/main/Deep%20Neural%20Networks/5%3A%20Shop%20Competition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<small><font color=gray>Notebook author: <a href="https://www.linkedin.com/in/olegmelnikov/" target="_blank">Oleg Melnikov</a> ©2021 onwards</font></small><hr style="margin:0;background-color:silver">

**<font size=6>🛒Shop</font>**. [**Instructions**](https://colab.research.google.com/drive/1riOGrE_Fv-yfIbM5V4pgJx4DWcd92cZr#scrollTo=ITaPDPIQEgXV) for running Colabs.

<details>
  <summary><small>Sharing consent: <mark>[ X ]</mark></summary>
  <div>
We consent to sharing our Colab (after the assignment ends) with other students/instructors for educational purposes. We understand that sharing is <b>optional</b> and this decision will not affect our grade in any way. <font color=gray><i>
Instructions: If ok with sharing your Colab for educational purposes, leave "X" in the check box.</i></font></small></div>

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')   # OK to enable, if your kaggle.json is stored in Google Drive

In [ ]:
!pip install inflect==7.0.0 >> log  # resolves pip's dependency issue for inflect 7.4.0 requires typeguard>=4.0.1
!pip install -U tensorflow_addons >> log
!pip install 'keras<3.0.0' mediapipe-model-maker >>log  # fix from https://github.com/google-ai-edge/mediapipe/issues/5229

In [ ]:
!mkdir -p ~/.kaggle                           # Kaggle exe uses kaggle.json in root's hidden .kaggle dir
!cp kaggle.json ~/.kaggle/kaggle.json >> log  # If kaggle.json in Colab's content dir (without GDrive)
!cp /content/drive/MyDrive/kaggle.json ~/.kaggle/kaggle.json >>log  # If kaggle.json is in GDrive's root
!chmod 600 ~/.kaggle/kaggle.json              # Only you have full read/write access to kaggle.json
!kaggle config set -n competition -v 17feb25-shop  # Set competition context for API calls heresince
!kaggle competitions download >> log          # Download competition data as a zip file
!unzip -o *.zip >> log                        # Unzip Kaggle data
!kaggle competitions leaderboard --show       # Print current public LB. See www.kaggle.com/docs/api

cp: cannot stat '/content/drive/MyDrive/kaggle.json': No such file or directory
- competition is now set to: 17feb25-shop
Using competition: 17feb25-shop
  teamId  teamName                    submissionDate       score         
--------  --------------------------  -------------------  ------------  
13401496  Nainika Devireddy           2025-03-02 16:38:13  0.9537600000  
13391697  shunguan                    2025-03-02 02:31:16  0.9527200000  
13419248  Oluwatobi Ajide             2025-03-02 12:24:56  0.9526800000  
13390955  David Bishop190             2025-03-02 00:20:37  0.9468000000  
13423329  JacobMarx                   2025-03-02 12:13:36  0.9393200000  
13397710  Sichao Liu                  2025-03-02 02:49:56  0.9349600000  
13392902  Yang Zhang                  2025-03-02 01:02:32  0.9229600000  
13395788  Group 13 Eric_Richard       2025-03-01 02:16:28  0.9220400000  
13396765  Shrinit Babel               2025-03-02 16:17:26  0.9194000000  
13413286  Arvin Ziaei           

In [ ]:
%%time
%%capture log_imports
%reset -f
from IPython.core.interactiveshell import InteractiveShell as IS; IS.ast_node_interactivity = "all"
import numpy as np, pandas as pd, time, matplotlib.pyplot as plt, seaborn as sns, tensorflow_addons as tfa
from sklearn.preprocessing import PolynomialFeatures
import tensorflow as tf, tensorflow.keras as keras
from keras.layers import Flatten, Dense
ToCSV = lambda df, fname: df.round(2).to_csv(f'{fname}.csv', index_label='ID') # rounds values to 2 decimals

class Timer():
  def __init__(self, lim:'RunTimeLimit'=60): self.t0, self.lim, _ = time.time(), lim, print(f'⏳ started. You have {lim} sec. Good luck!')
  def ShowTime(self):
    msg = f'Runtime is {time.time()-self.t0:.0f} sec'
    print(f'\033[91m\033[1m' + msg + f' > {self.lim} sec limit!!!\033[0m' if (time.time()-self.t0-1) > self.lim else msg)

np.set_printoptions(linewidth=100, precision=2, edgeitems=2, suppress=True)
pd.set_option('display.max_columns', 20, 'display.precision', 2, 'display.max_rows', 4)

CPU times: user 5.58 s, sys: 934 ms, total: 6.51 s
Wall time: 10.4 s


In [ ]:
df = pd.read_csv('XY_Shop.csv'); df

,Adm,AdmDur,Inf,InfDur,Prd,PrdDur,BncRt,ExtRt,PgVal,SpclDay,Mo,OS,Bsr,Rgn,TfcTp,VstTp,Wkd,Rev
0,0,0.00,0,0.0,18,132.99,3.82e-02,5.45e-02,0.0,0.0,4,3,1,1,2,0,1,NaN
1,1,0.00,0,0.0,37,1150.20,1.25e-03,3.03e-02,0.0,0.0,11,2,2,4,2,0,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499998,0,0.00,0,0.0,27,1185.14,0.00e+00,1.59e-03,0.0,0.0,5,2,2,2,3,0,1,0.0
499999,6,51.36,0,0.0,59,1898.21,0.00e+00,3.22e-03,0.0,0.0,12,2,2,2,1,0,0,0.0


In [ ]:
df.info()   # observe datatypes and any missing values

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 18 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   Adm      500000 non-null  int64  
 1   AdmDur   500000 non-null  float64
 2   Inf      500000 non-null  int64  
 3   InfDur   500000 non-null  float64
 4   Prd      500000 non-null  int64  
 5   PrdDur   500000 non-null  float64
 6   BncRt    500000 non-null  float64
 7   ExtRt    500000 non-null  float64
 8   PgVal    500000 non-null  float64
 9   SpclDay  500000 non-null  float64
 10  Mo       500000 non-null  int64  
 11  OS       500000 non-null  int64  
 12  Bsr      500000 non-null  int64  
 13  Rgn      500000 non-null  int64  
 14  TfcTp    500000 non-null  int64  
 15  VstTp    500000 non-null  int64  
 16  Wkd      500000 non-null  int64  
 17  Rev      450000 non-null  float64
dtypes: float64(8), int64(10)
memory usage: 68.7 MB


In [ ]:
vX = df.query('Rev!=Rev').drop('Rev', axis=1)  # slice a test sample, where revenue flag is empty
tXY = df.query('Rev==Rev')                     # slice training sample, where revenue flag is not empty (not NaN)
tX, tY = tXY.drop('Rev', axis=1), tXY.Rev      # split into training I/O
NumFeatures = list(tX.select_dtypes(include='float').columns)
print('Numeric features: ', NumFeatures)       # numeric/quantitative feature names

Numeric features:  ['AdmDur', 'InfDur', 'PrdDur', 'BncRt', 'ExtRt', 'PgVal', 'SpclDay']


In [ ]:
# def ScatterCorrHist(df):
#   def corrdot(*args, **kwargs):
#     # credit: https://stackoverflow.com/questions/48139899
#     corr_r = args[0].corr(args[1], 'pearson')
#     corr_text = f"{corr_r:2.2f}".replace("0.", ".")
#     ax = plt.gca();
#     ax.set_axis_off();
#     msz = abs(corr_r) * 5000   # marker size
#     fsz = abs(corr_r) * 40 + 5 # font size
#     ax.scatter([.5], [.5], msz, [corr_r], alpha=0.5, cmap='coolwarm', vmin=-1, vmax=1, transform=ax.transAxes)
#     ax.annotate(corr_text, [.5, .5,],  xycoords="axes fraction", ha='center', va='center', fontsize=fsz)

#   sns.set(style='white', font_scale=.8);
#   g = sns.PairGrid(df, aspect=1, diag_sharey=False);
#   g.fig.set_size_inches(20,10)
#   g.map_lower(sns.regplot, lowess=True, ci=False, line_kws={'color':'red'}, scatter_kws={'s':1});
#   g.map_diag(sns.histplot, kde_kws={'color':'black'});
#   g.map_upper(corrdot);
#   g.fig.suptitle("Scatter plot, Correlations and histograms on diagonal", y=1);
#   _ = plt.subplots_adjust(hspace=0.02, wspace=0.02);
#   _ = plt.show();

# df_ = tXY.loc[(tXY[NumFeatures]>0).all(axis=1), NumFeatures+['Rev']].sample(n=100, random_state=0)
# # df_ = df.select_dtypes(include='float').query('AdmDur>0 and InfDur>0 and PrdDur>0 and BncRt>0 and ExtRt>0 and PgVal>0 and SpclDay>0').sample(n=100, random_state=0)
# ScatterCorrHist(df_)

In [ ]:
tmr = Timer()

⏳ started. You have 60 sec. Good luck!


<hr color=green size=40>

<strong><font color=green size=5>⏳Timed Green Playground (TGP): Your ideas, code, documentation, and timer START HERE!</font></strong>

<font color=green>Students: Keep all your definitions, code, documentation in <b>TGP</b>. Modifying any code outside of TGP incurs penalties.

<font color=green><h3><b>Import Neccesary Libraries</b></h3></font>

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
import random
import os

<font color=green><h3><b>Seeding to Ensure Reproducibility</b></h3></font>

In [ ]:
# Ensure Reproducibility
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

<font color=green><h3><b>Feature Transformations</b></h3></font>

In [ ]:
tX['BncExtRatio'] = tX['BncRt'] / (tX['ExtRt'] + 1e-6)
vX['BncExtRatio'] = vX['BncRt'] / (vX['ExtRt'] + 1e-6)

tX['AvgDur'] = (tX['PrdDur'] + tX['AdmDur'] + tX['InfDur'] + 1) / (tX['Prd'] + tX['Adm'] + tX['Inf'] + 1e-6)
vX['AvgDur'] = (vX['PrdDur'] + vX['AdmDur'] + vX['InfDur'] + 1) / (vX['Prd'] + vX['Adm'] + vX['Inf'] + 1e-6)

In [ ]:
# Apply Log1p Transformations
transform_features = ['AdmDur', 'InfDur', 'PrdDur', 'BncRt', 'ExtRt', 'PgVal', 'SpclDay', 'AvgDur', 'BncExtRatio']
for feature in transform_features:
    if feature in tX.columns:
        tX[feature] = np.log1p(tX[feature])
        vX[feature] = np.log1p(vX[feature])

In [ ]:
# Create a mapping for categorical features based on frequency
for col in ['TfcTp', 'VstTp']:
  # Calculate frequency of each category
  frequency_map = tX[col].value_counts(normalize=True).to_dict()

  # Replace categories with their frequencies
  tX[col] = np.log1p(tX[col].map(frequency_map))
  vX[col] = np.log1p(vX[col].map(frequency_map))

<font color=green><h3><b>Stratified Sampling for Training</b></h3></font>

In [ ]:
# Reduce Training Set Size
tX_train, _, tY_train, _ = train_test_split(tX, tY, train_size=0.30, stratify=tY, random_state=SEED)

<font color=green><h3><b>Feature Scaling and Polynomial Features (degree = 3)</b></h3></font>

In [ ]:
pipeline = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("poly", PolynomialFeatures(degree=3, interaction_only=True, include_bias=False))
])

tX_scaled = pipeline.fit_transform(tX_train)
vX_scaled = pipeline.transform(vX)

<font color=green><h3><b>Define and Train Model</b></h3></font>

In [ ]:
model = keras.models.Sequential([
    keras.layers.Dense(200, activation="relu"),
    keras.layers.Dense(100, activation="relu"),
    keras.layers.Dense(100, activation="relu"),
    keras.layers.Dense(100, activation="relu"),
    keras.layers.Dense(1, activation='sigmoid')
])

lr_schedule = keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=0.001,
    decay_steps=5000,
    decay_rate=0.9
    )

model.compile(
    loss="binary_crossentropy",
    optimizer=keras.optimizers.Adam(learning_rate=lr_schedule),
    metrics=['accuracy'],
    jit_compile=True
    )

history = model.fit(
    tX_scaled, tY_train,
    validation_split=0.2,
    epochs=100,
    batch_size = 4000,
    verbose=0,
    callbacks=[keras.callbacks.EarlyStopping(patience=2)]
    )

<font color=green><h3><b>Make Predictions and Output Submission File</b></h3></font>

In [ ]:
# Make Predictions
pY = pd.DataFrame(model.predict(vX_scaled), index=vX.index+1, columns=['Rev'])

# Save Submission
ToCSV(pY.round(0).astype(int), 'attempt')

# Print Final Validation Accuracy
val_acc = history.history['val_accuracy']
print("Final Validation Accuracy:", val_acc[-1])


1563/1563 [==============================] - 3s 2ms/step
Final Validation Accuracy: 0.9545555710792542


<font color=green><h3><b>$\beta$. Idea Documentation</b></h3>
<details>
  <summary>Instructions</summary>
  <div>


1. **Audience**. Your peers who will learn from your Colab and ideas therein.
1. **Importance**. The ML/DL ideas are not entirely random, but are based on prior experience and systematized/organized experiments. We'd like students to share and learn from idea generation to idea experimentation process done in our class using tools learned thus far.
1. **Format**. Keep it concise/precise in consistent font/presentation. Include numbers/IDs to your References, such as [1] or [[Géron22]](https://scholar.google.com/scholar?cluster=498861685923226475), where these are defined in your References section below. This helps link your ideas/experiments to external ideas.
1. **Reproducibility**. Your description should contain reasonable details needed for reproducibility, i.e. describe the state of your modeling pipeline before the change is made, what is changed and how the idea was discovered, and what improvement it resulted in. Thus, peers can try this idea with an expectation of the value it brings. See examples below.
1. **Bonus** points for the exceptional/exemplary/educational documentation (see grading rubric).
****
1. **TODO**: Describe the key idea in your work in the following format (similar to a "micro publication"):
  1. **Title**. Give each idea a descriptive name (i.e. a micro abstract).
    1. Ex(ample). <i>"Thresholding carat feature outliers improves MAE by 3% on public LB"</i>
  1. **Idea Discovery**. What led you to this idea? Was it some [EDA](https://en.wikipedia.org/wiki/Exploratory_data_analysis), familiarity with this dataset or some of the features?
    1. Ex. <i>"We plotted all univariate distributions of variables and discovered that diamond carat had unreasonable (but rare) values below and above [0,10] interval, when plotted carat's histogram in the train and test sets, which contained 10 and 3 such outliers respectively. We decided to use 10 as a reasonable threshold because it is 99th percentile of carat values in the 20K baseline sample. See our histogram plot below [plot here]. "</i>
  1. **Finding's Importance**. Describe why you think the idea was important to proceed with.
    1. Ex. <i>"We use a linear model, the slope of which is sensitive to outliers on the periphery of the feature space domain. The fitted hyperplane slopes in the direction of the extreme training feature values thereby mapping a non-existent relation between carat size and diamond price, which is not expected to repeat in the test set. "</i>
  1. **Experiment Setup**.
  How did you set up experiments to test your idea? What resources were helpful? What metric did you select, why and what values did you observe?
    1. Ex. <i>"To alleviate the impact of the outlying feature values, we need to either remove observations with extreme values, or somehow cap them (to stay within the distribution of the other carat values) or use a model insensitive to outliers (such as robust regression). We learned 3 suitable methods for treating outliers in [ref]: ... [It'd be great to briefly describe each method] We tried each one on a Baseline model, while keeping the competition-required [MAE](https://en.wikipedia.org/wiki/Mean_absolute_error) metric. We tested each method locally on the seeded 50/50 split of the 20K training set sampled in baseline Colab."</i>
  1. **Results**. What was the result or metric improvement from implementing the experiment locally and/or on public LB?
    1. Ex. <i>"Baseline MAE was 539.1257546465 in public LB and 530 in local default experiment with 50/50 train-test split. When applied on the same-seed split, Methods 1,2,and 3 showed 1%, 2%, and 5% improvement on the test set. When uploaded to public LB, Method 3 showed a 3% improvement. So, we decided to keep method 3."</i>

</div> </details>
</font>


<font color=green><h4><b>Task 1. Preprocessing Ideas</b></h4>
<details>
  <summary>Instructions</summary>
  <div>Explain a <b>key idea</b> that helped in <b>preprocessing pipeline</b>. This may be about some feature engineering, tricky subsampling, clustering, dimension reduction, etc. Use the format in TODO specified above. Remember to provide citation references for the peers to read more into your work.
</div> </details>
</font>

1. **Title**:
Feature Transformation (Logarithmic Transformations and Frequency Encoding)
1. **Idea Discovery**:
Initial data exploration showed that several continuous numerical features showed heavily right-skewed distribution, which increased the potential risk of bias towards the more frequent values and the dominating class. To address this, logarithmic and square root transformations were explored to stabilize the variance and improve feature interpretability. Similarly, categorical variables, such as Traffic Type (TfcTp) and Visit Type (VstTp), were evaluated to find transformations that emphasized class separation. One-hot encoding and frequency encoding were considered for implementation. Frequency encoding involves replacing categorical labels with their respective normalized frequencies in the dataset. The hypothesis was that frequency encoding would improve efficiency and emphasize distinctions between the two classes more effectively than one-hot encoding, which can introduce sparsity and redundant dimensions.
1. **Finding's Importance**:
Feature transformation and encoding directly impact model performance, especially in high-dimensional datasets. Feature transformation can enhance model performance by redistributing skewed data closer to a normal distribution, improving the learning algorithm's ability to detect patterns [1]. Logarithmic and square root transformations help stabilize variance, normalize distributions and reduce the impact of extreme values, leading to feature relationships [1]. The log1p transformation (log(1 + x)) was particularly considered because it effectively handles zeros and small values. The square root transformation, while less aggressive, also helps manage skewness by compressing large values. Frequency encoding also aimed to introduce a continuous representation that could better depict the influence of less common categories in the feature.
1. **Experiment Setup**:
Each feature transformation was tested within the model pipeline, using a 30% stratified training split to endure balanced class representation. Models were trained with and without transformations, and mean validation accuracy was compared. Logarithmic (log1p) and square root (sqrt) transformations were applied to each of the continuous variables individually and compared to the baseline. Categorical features such as Traffic Type, Visit Type, OS, Browser, Region, Weekend and Month were encoded based on frequency mapping to reflect their relative importance, or one-hot encoded and compared to the baseline.
1. **Results**:
The Log1p transformation led to the highest validation accuracy, outperforming both the square root and baseline methods. In particular, the log transformation of all of the duration features (AdmDur, InfDur, PrdDur), Bounce Rate (BncRt), Exit Rate (ExtRt), Page Value (PgVal) and Special Day (SpclDay) increased accuracy, while any transformation of Administrative (Adm), Informational (Inf), Product Related (Prd) features decreased accuracy. Frequency encoding provided a consistent improvement when applied to Traffic Type and Visit Type, suggesting that the distribution of user types influenced the target variable. One-hot encoding declined performance, highlighting that not all categorical variables contribute equally to model efficacy. Any transformations made to OS, Browser, Region, Weekend and Month resulted in decreased accuracy, so they were left as is.


1. **Title**:
Scaling and Polynomial Feature Engineering
1. **Idea Discovery**:
Scaling and normalization are critical in machine learning, especially when dealing with algorithms sensitive to feature magnitude, such as neural networks. Standard scaling (Z-score normalization) was chosen to transform features to a standardized scale with a mean of 0 and a standard deviation of 1. Polynomial feature engineering was implemented to introduce higher-order interactions between features.
1. **Finding's Importance**:
Proper scaling prevents certain features from dominating the learning process due to differences in scale, which is particularly important when combining features of varied units and ranges. Polynomial features help models learn complex patterns by enabling interaction terms that may not be apparent in the original data [1]. However, limiting overexpansion of the feature set is crucial to avoid the curse of dimensionality that could lead to overfitting and increased computational costs [1].
1. **Experiment Setup**:
StandardScaler was applied to the entire dataset before model training. Polynomial features up to the third degree were generated using the PolynomialFeatures class with interaction_only set to True, preventing overly complex feature combinations. Polynomial feature degrees 1, 2, and 3 were evaluated. Validation accuracy was compared with and without scaling and polynomial transformations. The impact on training speed and validation stability was also assessed.
1. **Results**:
Standard scaling consistently improved model stability. Polynomial feature engineering with a degree of 3 provided the best trade-off between model complexity and performance. Lower degrees did not capture enough non-linear relationships. While a degree of 3 drastically increased runtime, the increased performance by 10% validated the choice to proceed with it.

1. **Title**:
Feature Selection and Engineering
1. **Idea Discovery**:
Apart from transforming the existing features, new features were engineered to enhance the model. The Bounce vs. Exit Rate ratio was developed to quantify how often users left the site immediately after visiting a page, compared to those who exited after navigating through additional content. The Average Duration feature was introduced to measure the average time spent on the site, considering different types of durations (Product, Administrative, and Informational). These new features aimed to describe user behavior more intently than the existing features. Within the original features, it is also important to identify features that do not contribute to the predictive power of the model. The deletion of existing variables was also explored.
1. **Finding's Importance**:
Creating derived features allows models to tap into patterns in the dataset that may not have been apparent previously. Behavioral metrics can be strong predictors for a customer’s intent to purchase, hence potentially improving classification accuracy. Removing features that do not contribute to the model’s performance allows the model to focus on key features, while also reducing feature dimensionality. As a result, there may be an increase in performance and efficiency, and a decrease in runtime.
1. **Experiment Setup**:
The derived features were added to the dataset, and models were retrained to measure their impact on validation accuracy. These engineered features were tested individually and in combination to assess their individual contributions to performance improvements. Model performance with new features was compared to a baseline model. Similarly, each of the original features was removed from the dataset individually and compared to the baseline to determine their contribution to model prediction.
1. **Results**:
The Bounce vs. Exit Rate ratio provided a substantial boost to validation accuracy (3%), reinforcing the hypothesis that user exit patterns are predictive of the target variable. Average Duration also contributed positively, with a smaller effect of 0.5%. Combining both derived features with logarithmic transformations provided the highest accuracy. Removing any of the original features decreased accuracy of the model, suggesting that each of the features contributed to the significantly to the predictive power.

1. **Title**:
Dimensionality Reduction
1. **Idea Discovery**:
Given the high-dimensional nature of the feature space after polynomial expansion (degree = 3), dimensionality reduction techniques were explored to improve computational efficiency. Principal Component Analysis (PCA) was initially selected due to its effectiveness in capturing variance while reducing redundancy in the data. However, alternative methods such as Incremental PCA, Randomized PCA, and Linear Discriminant Analysis (LDA) were also evaluated to determine whether a faster and more effective dimensionality reduction method could be applied.
1. **Finding's Importance**:
Reducing feature dimensionality is crucial for deep learning models, as excessive input dimensions can lead to high computational costs, increased memory usage, and a risk of overfitting [1]. While PCA allows for better capturing of the data variance, Incremental PCA and Randomized PCA potentially offer a more memory and time efficient approach. LDA was also explored, as it finds the directions that maximize class separability rather than variance.
1. **Experiment Setup**:
Each dimensionality reduction method was applied to the polynomial-expanded dataset, and its impact on training time and validation accuracy was measured. PCA was tested at various explained variance thresholds (ranging from 95% to 99.9%) to determine the optimal number of retained components. Incremental PCA was evaluated in different batch sizes to balance computational speed and variance preservation. Randomized PCA was tested for speed improvements while maintaining competitive accuracy. LDA was applied as a supervised alternative, using class labels to maximize separability.
1. **Results**:
Standard PCA with 99.9% explained variance provided the best performance but was computationally slow, taking over a minute to run. Incremental PCA and Randomized PCA reduced memory consumption but still required substantial processing time, more than the Standard PCA. LDA failed to outperform PCA, as it struggled to capture complex feature relationships beyond simple linear class separation, dropping accuracy to 73%. Initially, standard PCA with 99.9% explained variance was retained. However, with further optimization of the model architecture, the impact of PCA diminished while increasing time complexity. It was ultimately removed from the model.

<font color=green><h4><b>Task 2. Modeling Ideas</b></h4>
<details>
  <summary>Instructions</summary>
  <div>Explain a <b>key idea</b> that helped with <b>model selection</b> in the format specified above. This may include tuning model parameters (perhaps a grid search with specific parameter range) or some other experiments, search/choice of the suitable model, experiments with postprocessing of model predictions, etc. Use the format in TODO specified above. Remember to provide citation references for the peers to read more into your work.
</div> </details>
</font>

1. **Title**:
Model Architecture and Optimization
1. **Idea Discovery**:
Deep learning models were optimized through architecture tuning, activation function selection, and learning rate scheduling. The primary objective was to enhance model stability while maintaining computational efficiency, ensuring that training remained under 60 seconds without sacrificing performance. Regularization techniques such as L2 weight decay and dropout were explored to prevent overfitting. Various optimizers, including Adam, AdamW, and SGD, were tested to improve convergence speed and generalization. Network depth and node count were also finetuned for the number of hidden layers and units per layer.
1. **Finding's Importance**:
Optimizing model architecture is crucial for achieving high accuracy without excessive complexity. A well-balanced architecture prevents overfitting to the training set, ensures fast and stable learning and allows for smooth convergence. The goal is to prevent computational complexity, while also designing an architecture that captures the complex relationships within the dataset [1].
1. **Experiment Setup**:
To efficiently identify the most optimal deep learning architecture, Keras Tuner’s Hyperband search was used to explore network depth, node count per layer, activation functions, learning rate scheduling, regularization techniques, and optimizers in a time-efficient manner. The network depth and node count was optimized between 2 to 5 hidden layers and 32 to 256 nodes per layer. Activation function such as ReLU, Swish, ELU and GeLU were tested. Regularization techniques such as L2 weight decay (0.0001 to 0.001) and dropout (0.1 to 0.4) were tested. Learning rate schedulers, such as exponential decay and cosine decay were evaluated to optimize a smoother convergence. The optimizer choice was tuned from Adam, AdamW and SGD. Batch sizes during training were evaluated between 32 and 8000. Early stopping (1, 2, 3, 4, 5) was explored to terminate training when validation performance stabilized.
1. **Results**:
The final optimized model, selected through Keras Tuner's Hyperband search, consisted of four hidden layers with neuron counts of 200, 100, 100, and 100, respectively. ReLU activation was chosen for all layers due to its strong convergence properties and consistent performance across trials. Among the learning rate schedulers tested, exponential decay with an initial learning rate of 0.001 and a decay rate of 0.9 provided the most stable convergence. Adam emerged as the best-performing optimizer, offering both speed and robustness in training. The optimal batch size was determined to be 4000, striking a balance between computational efficiency and model stability. Early stopping (patience=2) successfully prevented overfitting while ensuring minimal runtime. The final model achieved a validation accuracy of 95.46%, outperforming all previously tested configurations while maintaining a total runtime of under 60 seconds.

<font color=green><h3><b>$\gamma$. References</b></h3>
<details>
  <summary>Instructions</summary>
  <div>

1. Cite your sources to help your peers learn from these (and to avoid plagiarism).
1. HOML textbook should be cited, since we used it in this week's learning.
1. Use Google Scholar to draw [APA](https://en.wikipedia.org/wiki/American_Psychological_Association) citation format for books and publications.
1. Cite [StackOverflow](https://stackoverflow.com/), YouTube videos, package docs, open-access textbooks/publicaitons and other meaningful internet resources that you used.
1. We may reward exceptional and meaningful citations (not just a list of [SKL](https://scikit-learn.org/stable/)/[TF](https://www.tensorflow.org/) manual pages and a list of articles.) For example, if you used an idea from a publication, indicate it in TGP with a number that corresponds to its reference in References.

</div> </details>
</font>

1. Geron, A. (2019). Hands-On Machine Learning with Scikit-Learn, Keras & Tensorflow. 2nd Ed., Sebastopol, CA: O’Reilly, 2019.


<font size=5>⌛</font> <strong><font color=green size=5>Do not exceed competition's runtime limit! Do not write code outside TGP</font></strong>
<hr color=green size=40>

In [ ]:
tmr.ShowTime()    # measure Colab's runtime. Do not remove. Keep as the last cell in your notebook.

Runtime is 55 sec


<details>
  <summary><font size=5><b>💡Starter Ideas</b></font></summary>
  <div>
  
**Model**
1. Tune model hyperparameters, batch size, optimizer, NN layers

**Features**
1. Try to linear and non-linear feature normalization: shift/scale, log, divide features by features (investigate scatterplot matrix)
1. Try higher order feature interactions and polynomial features on a small subsample. Then identify key features or select key principal components. The final model can be trained on a larger or even full training sample. You can use [PCA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html) to reduce the feature set
1. Incorporate categorical features (appropriately encoded)
  1. E.g. you could replace codes (or groups of codes) with their frequencies, which may capture the implied "distance" or rarity between category levels.
  1. If encoding ordinal features with integers, should non-equidistant values be considered?

**Training observations**
1. Try clustering methods to remove similar observations. You may also try dimension reduction methods (eg. PCA) on the transposed data matrix (if it has scaled numeric features).
1. Look for and deal with outliers or influential points in the training set
1. Deal with **imbalanced sample**: oversample smaller class, or undersample larger class, or provide observation weights or provide class weights, or seek a suitable loss function
1. Investigate distributions of features. Any missing values? Any zero values?

**Predictions**
1. Evaluate predictions and focus on poorly predicted "groups":
  1. Strongest misclassifications. E.g. the model is very confident about the wrong label
  1. Evaluate predictions near decision boundaries.

**EDA and Domain Expertise**
1. Do a thorough EDA: look for feature augmentations that result in linear decision boundaries between pairs of classes.
1. Learn about the domain: how should output relate to features? How do month or weekend impact users' buying activity?
  1. User Agent [&#127910;](https://www.youtube.com/results?search_query=user+agent+browser), Google Analytics [&#127910;](https://www.youtube.com/results?search_query=google+analytics), tracking online shopping intent [&#127910;](https://www.youtube.com/results?search_query=tacking+online+shopping+intent), [📄](https://scholar.google.com/scholar?q=tracking+online+shopping+intent)

</div> </details>